In [2]:
import struct
import numpy as np
import matplotlib.pyplot as plt
import os
from collections import namedtuple

HEADER_SIZE = 17   # "BLENDER17-01v0501" (new large-header format)
BHEAD = struct.Struct("<4si Q q q")  # code, SDNAnr, old, len, nr = 32 bytes
GLOB_struct = struct.Struct("<4s 3h 6x 4Q 2i Q 16s 1024s 64s 9f 12x")
assert GLOB_struct.size == 1216

path = os.path.expanduser("/mnt/vfx_library/developement_projects/YOUTUBE/blender/blender DNA/structures_uncompressed.blend")

accumulate_byte = HEADER_SIZE

# buffer containing the explained data blocks, as human-readable text
out_file_buffer = ""

def append_to_buffer(label, value):
    # append a "label: value" line to the global text buffer
    global out_file_buffer
    if isinstance(value, bytes):
        # show raw bytes as their decoded string when possible, else as hex
        try:
            shown = value.decode('utf-8').rstrip('\x00')
        except UnicodeDecodeError:
            shown = value.hex()
    else:
        shown = value
    out_file_buffer += f"{label}: {shown}\n"

def buffer_new_line():
    global out_file_buffer
    out_file_buffer += "\n"

def save_buffer_to_file(filename):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(out_file_buffer)

def print_header(code, sdna, old, length, nr):
    print(f"Code: {code}, \nSDNA: {sdna}, \nOld: {old}, \nLength: {length}, \nNR: {nr}")


FileGlobal = namedtuple("FileGlobal", [
    "subvstr", "subversion", "minversion", "minsubversion",
    "curscreen", "curscene", "cur_view_layer", "pad1",
    "fileflags", "globalf", "build_commit_timestamp",
    "build_hash", "filepath", "colorspace_name", *[f"m{i}" for i in range(9)],
])

def read_glob(payload: bytes) -> FileGlobal:
    v = list(GLOB_struct.unpack(payload))
    v[0]  = v[0].decode()
    v[11] = v[11].rstrip(b"\0").decode()
    v[12] = v[12].rstrip(b"\0").decode()
    v[13] = v[13].rstrip(b"\0").decode()
    return FileGlobal(*v)



with open(path, "rb") as f:
    header = f.read(HEADER_SIZE)
    if not header.startswith(b"BLENDER"):
        raise ValueError("Not a plain .blend (maybe compressed?): %r" % header[:12])
    else:
        append_to_buffer("Header", header)
        buffer_new_line()
        print(f"header content: {header}")
        print("-"*40)
        print(" ")

    while True:
        # read 32 bytes
        raw = f.read(BHEAD.size)

        # make sure the size is correct
        if len(raw) < BHEAD.size:
            print("Reached end of file or incomplete header, stopping.")
            break

        # use python struct.unpack method
        code, sdna, old, length, nr = BHEAD.unpack(raw)
        append_to_buffer("Code", code)
        append_to_buffer("SDNA", sdna)
        append_to_buffer("Old", old)
        append_to_buffer("Length", length)
        append_to_buffer("NR", nr)
        print_header(code, sdna, old, length, nr)

        # skip bytes inside content to get the next header
        content = f.read(length)
        accumulate_byte+= BHEAD.size + length

        # specific data block handling
        if code == b"REND":
            # first 4 bytes are start frame, next 4 bytes are final frame
            start_frame, final_frame = struct.unpack("<ii", content[:8])
            append_to_buffer("Start frame", start_frame)
            append_to_buffer("Final frame", final_frame)
            print(f"Render block: start_frame={start_frame}, final_frame={final_frame}")

            # The rest of the content is the scene name
            scene_name = content[8:].decode('utf-8').rstrip('\x00')
            append_to_buffer("Scene name", scene_name)
            print(f"Scene name: {scene_name}")

        elif code == b"TEST":
            # The TEST block contains the thumbnail image data
            # The first 4 bytes are width, next 4 bytes are height
            width, height = struct.unpack("<ii", content[:8])
            append_to_buffer("Thumbnail width", width)
            append_to_buffer("Thumbnail height", height)
            print(f"Thumbnail size: {width}x{height}")

            # The rest of the content is the pixel data (RGBA)
            pixels = np.frombuffer(content[8:], dtype=np.uint8)
            img = pixels.reshape(height, width, 4)
            print(f"Thumbnail image shape: {img.shape}")

        elif code == b"GLOB":
            # The GLOB block contains global data
            g = read_glob(content)
            append_to_buffer("Subversion string", g.subvstr)
            append_to_buffer("Subversion", g.subversion)
            append_to_buffer("Min version", g.minversion)
            append_to_buffer("Min subversion", g.minsubversion)
            append_to_buffer("Current screen (old ptr)", hex(g.curscreen))
            append_to_buffer("Current scene (old ptr)", hex(g.curscene))
            append_to_buffer("Current view layer (old ptr)", hex(g.cur_view_layer))
            append_to_buffer("File flags", hex(g.fileflags))
            append_to_buffer("Global flags", hex(g.globalf))
            append_to_buffer("Build commit timestamp", g.build_commit_timestamp)
            append_to_buffer("Build hash", g.build_hash)
            append_to_buffer("Color space", g.colorspace_name)
            print(f"GLOB: version {g.subversion}, hash {g.build_hash}, {g.colorspace_name}")


        else:
            print(f"Unknown code: {code}, skipping {length} bytes.")

        # accumulated bytes in the buffer
        append_to_buffer("Accumulated bytes", accumulate_byte)

        print(f"Accumulated bytes: {accumulate_byte}")
        print("-" * 40)
        print(" ")

        buffer_new_line()

# save text file with the same name as the binary file
save_buffer_to_file("extracted_blend_data.txt")


header content: b'BLENDER17-01v0502'
----------------------------------------
 
Code: b'REND', 
SDNA: 0, 
Old: 16, 
Length: 264, 
NR: 1
Render block: start_frame=1, final_frame=511
Scene name: Scene.001
Accumulated bytes: 313
----------------------------------------
 
Code: b'TEST', 
SDNA: 0, 
Old: 8918696635957482208, 
Length: 59912, 
NR: 1
Thumbnail size: 117x128
Thumbnail image shape: (128, 117, 4)
Accumulated bytes: 60257
----------------------------------------
 
Code: b'GLOB', 
SDNA: 171, 
Old: 8918696635957482224, 
Length: 1216, 
NR: 1
GLOB: version 44, hash fbe6228777e7, Linear Rec.709
Accumulated bytes: 61505
----------------------------------------
 
Code: b'WM\x00\x00', 
SDNA: 972, 
Old: 5746685826338512976, 
Length: 1496, 
NR: 1
Unknown code: b'WM\x00\x00', skipping 1496 bytes.
Accumulated bytes: 63033
----------------------------------------
 
Code: b'DATA', 
SDNA: 973, 
Old: 1677542049005415184, 
Length: 248, 
NR: 1
Unknown code: b'DATA', skipping 248 bytes.
Accumulated b